In [230]:
tkrs = [
    "ADBE",
    "AVY",
    "AZO",
    "BTU",
    "CCU",
    "CMG",
    "CTAS",
    "FITB",
    "GME",
    "HOG",
    "PH",
    "WU",
]
pos = [-1, -1, -1, 1, 1, -1, -1, 1, 1, 1, -1, 1]

In [224]:
import pandas as pd
import numpy as np
import yfinance as yf

tkr_data = yf.Tickers(tkrs).history(start="2024-04-01")
tkr_d = {
    idx: gp.xs(idx, level=0, axis=1)
    for idx, gp in tkr_data.swaplevel(axis=1).groupby(level=0, axis=1)
}

for tkr, df in tkr_d.items():
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"]) - np.log(df["Close"].shift(1))

[*********************100%***********************]  12 of 12 completed
/tmp/ipykernel_74581/2857270705.py:8: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  for idx, gp in tkr_data.swaplevel(axis=1).groupby(level=0, axis=1)


In [225]:
from pandas_datareader.famafrench import get_available_datasets
import pandas_datareader.data as web

ds = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench")

/tmp/ipykernel_74581/669063850.py:4: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ds = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench")


In [226]:
factors = ds[0].loc["2024-04-01":]
factors.index = factors.index.tz_localize("UTC")
factors

,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
2024-04-01 00:00:00+00:00,-0.27,-0.91,-0.27,0.34,-0.31,0.021
2024-04-02 00:00:00+00:00,-0.85,-1.03,0.04,0.26,0.02,0.021
2024-04-03 00:00:00+00:00,0.16,0.33,-0.04,0.10,-0.19,0.021
2024-04-04 00:00:00+00:00,-1.23,0.20,0.18,0.03,0.10,0.021
2024-04-05 00:00:00+00:00,1.05,-0.50,-0.69,-0.29,-0.59,0.021
...,...,...,...,...,...,...
2024-08-26 00:00:00+00:00,-0.34,0.33,0.16,0.13,-0.06,0.022
2024-08-27 00:00:00+00:00,0.06,-0.90,0.02,0.27,0.23,0.022
2024-08-28 00:00:00+00:00,-0.67,-0.22,1.14,0.55,-0.16,0.022


In [228]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

results = {}
for tkr, df in tkr_d.items():
    reg_df = df.join(factors)
    reg_df = reg_df.rename({"Mkt-RF": "MktRF"}, axis="columns")
    mod = smf.ols(formula="returns ~ MktRF + SMB + HML + RMW + CMA", data=reg_df)
    res = mod.fit()
    results[tkr] = res.params

result_df = pd.DataFrame(results)
hedged_df = result_df * pos
pd.DataFrame({"result": result_df.sum(axis=1), "hedged": hedged_df.sum(axis=1)})

,result,hedged
Intercept,0.010576,-0.010576
MktRF,0.104486,-0.104486
SMB,0.070562,-0.070562
HML,-0.005840,0.005840
RMW,0.044221,-0.044221
CMA,-0.008643,0.008643
